In [13]:
import pandas as pd

In [14]:
# 1) Memuat data
file_path = "Sales Transaction v.4a.csv"
df = pd.read_csv(file_path, sep=';')
df.head()

,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country
0,581482,12/09/2019,22485,Set Of 2 Wooden Market Crates,21.47,12,17490.0,United Kingdom
1,581475,12/09/2019,22596,Christmas Star Wish List Chalkboard,10.65,36,13069.0,United Kingdom
2,581475,12/09/2019,23235,Storage Tin Vintage Leaf,11.53,12,13069.0,United Kingdom
3,581475,12/09/2019,23272,Tree T-Light Holder Willie Winkie,10.65,12,13069.0,United Kingdom
4,581475,12/09/2019,23239,Set Of 4 Knick Knack Tins Poppies,11.94,6,13069.0,United Kingdom


In [15]:
# 2) Standarisasi nama kolom
expected_cols = [
    "TransactionNo", "Date", "ProductNo", "ProductName",
    "Price", "Quantity", "CustomerNo", "Country"
]
df.columns = [col.strip() for col in df.columns]
print(df.columns.tolist())

['TransactionNo', 'Date', 'ProductNo', 'ProductName', 'Price', 'Quantity', 'CustomerNo', 'Country']


In [16]:
# 3) Menghapus spasi berlebih pada kolom teks
text_cols = ["TransactionNo", "ProductNo", "ProductName", "CustomerNo", "Country"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

In [17]:
# 4) Memperbaiki typo dan normalisasi nilai Country
if "Country" in df.columns:
    df["Country"] = df["Country"].replace({"United King dom": "United Kingdom", "United King": "United Kingdom"})
    df["Country"] = df["Country"].str.replace(r"\s+", " ", regex=True).str.strip()

df["Country"].value_counts().head() if "Country" in df.columns else None

Country
United Kingdom    485095
Germany            10675
France             10526
EIRE                8048
Belgium             2539
Name: count, dtype: int64

In [18]:
# 5) Konversi tipe data secara aman
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
if "Price" in df.columns:
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
if "Quantity" in df.columns:
    df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")

for col in ["TransactionNo", "ProductNo", "CustomerNo"]:
    if col in df.columns:
        df[col] = df[col].replace({"nan": pd.NA, "": pd.NA})

df.dtypes

TransactionNo            object
Date             datetime64[ns]
ProductNo                object
ProductName              object
Price                   float64
Quantity                  int64
CustomerNo               object
Country                  object
dtype: object

In [19]:
# 6) Menghapus baris duplikat yang identik
before_dupes = len(df)
df = df.drop_duplicates().copy()
after_dupes = len(df)

print(f"Jumlah baris sebelum hapus duplikat: {before_dupes}")
print(f"Jumlah baris sesudah hapus duplikat: {after_dupes}")

Jumlah baris sebelum hapus duplikat: 536350
Jumlah baris sesudah hapus duplikat: 531150


In [20]:
# 7) Menangani missing value pada kolom penting
critical_cols = ["TransactionNo", "Date", "ProductNo", "ProductName", "Price", "Quantity", "CustomerNo", "Country"]
existing_critical = [c for c in critical_cols if c in df.columns]

before_missing = len(df)
df = df.dropna(subset=existing_critical).copy()
after_missing = len(df)

print(f"Jumlah baris sebelum drop missing: {before_missing}")
print(f"Jumlah baris sesudah drop missing: {after_missing}")

Jumlah baris sebelum drop missing: 531150
Jumlah baris sesudah drop missing: 531095


In [21]:
# 8) Pembersihan berdasarkan aturan bisnis
if "Quantity" in df.columns:
    df = df[df["Quantity"] > 0]
if "Price" in df.columns:
    df = df[df["Price"] > 0]

print(f"Jumlah baris setelah aturan bisnis: {len(df)}")

Jumlah baris setelah aturan bisnis: 522601


In [22]:
# 9) Rekayasa fitur untuk analisis lanjutan
if {"Price", "Quantity"}.issubset(df.columns):
    df["TotalAmount"] = df["Price"] * df["Quantity"]
if "Date" in df.columns:
    df["YearMonth"] = df["Date"].dt.to_period("M").astype(str)

df[["Price", "Quantity", "TotalAmount", "YearMonth"]].head() if {"Price", "Quantity", "TotalAmount", "YearMonth"}.issubset(df.columns) else df.head()

,Price,Quantity,TotalAmount,YearMonth
0,21.47,12,257.64,2019-12
1,10.65,36,383.40,2019-12
2,11.53,12,138.36,2019-12
3,10.65,12,127.80,2019-12
4,11.94,6,71.64,2019-12


In [23]:
# 10) Perapihan tipe data akhir
if "Quantity" in df.columns:
    df["Quantity"] = df["Quantity"].astype(int)

for col in ["TransactionNo", "ProductNo", "ProductName", "CustomerNo", "Country"]:
    if col in df.columns:
        df[col] = df[col].astype("category")

df.dtypes

TransactionNo          category
Date             datetime64[ns]
ProductNo              category
ProductName            category
Price                   float64
Quantity                  int64
CustomerNo             category
Country                category
TotalAmount             float64
YearMonth                object
dtype: object

In [24]:
# 11) Menyimpan dataset hasil cleaning
cleaned_file = "Sales_Transaction_v4a_cleaned.csv"
df.to_csv(cleaned_file, index=False)

print(f"Dataset bersih tersimpan di: {cleaned_file}")

Dataset bersih tersimpan di: Sales_Transaction_v4a_cleaned.csv


In [25]:
# 12) Ringkasan hasil cleaning
print("=== RINGKASAN DATA CLEANING ===")
print(f"Jumlah baris sebelum hapus duplikat: {before_dupes}")
print(f"Jumlah baris sesudah hapus duplikat: {after_dupes}")
print(f"Jumlah baris sebelum drop missing: {before_missing}")
print(f"Jumlah baris sesudah drop missing: {after_missing}")
print(f"Jumlah baris final: {len(df)}")
print("\nMissing value setelah cleaning:")
print(df.isna().sum())

df.head()

=== RINGKASAN DATA CLEANING ===
Jumlah baris sebelum hapus duplikat: 536350
Jumlah baris sesudah hapus duplikat: 531150
Jumlah baris sebelum drop missing: 531150
Jumlah baris sesudah drop missing: 531095
Jumlah baris final: 522601

Missing value setelah cleaning:
TransactionNo    0
Date             0
ProductNo        0
ProductName      0
Price            0
Quantity         0
CustomerNo       0
Country          0
TotalAmount      0
YearMonth        0
dtype: int64


,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country,TotalAmount,YearMonth
0,581482,2019-12-09,22485,Set Of 2 Wooden Market Crates,21.47,12,17490.0,United Kingdom,257.64,2019-12
1,581475,2019-12-09,22596,Christmas Star Wish List Chalkboard,10.65,36,13069.0,United Kingdom,383.40,2019-12
2,581475,2019-12-09,23235,Storage Tin Vintage Leaf,11.53,12,13069.0,United Kingdom,138.36,2019-12
3,581475,2019-12-09,23272,Tree T-Light Holder Willie Winkie,10.65,12,13069.0,United Kingdom,127.80,2019-12
4,581475,2019-12-09,23239,Set Of 4 Knick Knack Tins Poppies,11.94,6,13069.0,United Kingdom,71.64,2019-12
